### Bag of Stories
**Description:** Compute segment similarity in a set of stories using sentence-transformers.

**Contributor:** Florentina Armaselu  

In [11]:
# Import required packages
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import datetime as dt
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import numpy as np


Define the input, output paths.

In [12]:
# Define the base directory relative to the current script location
base_dir = os.getcwd()
# Set the path to the input data folder for whole stories
input = os.path.join(base_dir, "./../data/grimm/")
# Set the path to the input data folder for segmented stories with TextTiling
input_seg_tt = os.path.join(base_dir, "./../data/annotated_stories/grimm/text-tiling/")
# Set the path to the input data folder for segmented stories with LLM episodes
input_seg_llm = os.path.join(base_dir, "./../data/annotated_stories/grimm/llm-episodes/")
# Set the path to the output folder for SGS results on the input stories
output = os.path.join(base_dir, "./../results/seg-sim/grimm/")
# Set the path of the output folder for the intermediate results
output_inter_res = os.path.join(base_dir, "./../results/intermediate_results/grimm/")

Define constants.

In [13]:
# String constants for SGS types
SGS_ADJACENT = 'adjacent'
SGS_CONTEXTUAL = 'contextual'

# String constants for segmentation methods
SEG_EQUAL_LENGTH = 'equal-length'
SEG_TEXT_TILING = 'text-tiling'
SEG_LLM_EPISODE = 'llm-episode'

# String constants for annotation elements
ANN_SEG = 'Segment' # Label for segments
ANN_SEG_PAIR = 'Segment Pair' # Label for segment pairs
ANN_SEP = '-'*40 # Separator for segments

# String constants for output subfolders and file formats
OUT_CSV = 'csv'
OUT_PNG = 'png'

# String constants for file naming
FN_ADJACENT = 'adj'
FN_CONTEXTUAL = 'ctx'
FN_EQUAL_LENGTH = 'el'
FN_TEXT_TILING = 'tt'
FN_LLM_EPISODE = 'llmep'

# String constant for seg-sim (SGS) labels
SGS_LABEL = 'SGS'
SGS_AVG = 'SGS Avg'
SGS_STD = 'SGS Std'
SGS_DEV = 'SGS Dev'

# String constants for types of intermediate results
INTER_RES_SEG = 'segment_check' # Intermediate results for segment check
INTER_RES_METH = 'method_check' # Intermediate results for method check

Define the model to be used for computing segment similarities.

In [14]:
# Load the model for segment similarity computation
# Using a pre-trained SentenceTransformer model for semantic similarity
model = SentenceTransformer('all-MiniLM-L6-v2')

Define functions for story segmentation.

In [15]:
# Divide stories into segments of approximately equal length in number of tokens
def el_segment_story(story, seg_length=100):
    """
    Segments a story into smaller parts of approximately equal length in tokens.
    
    Args:
        story (str): The full text of the story.
        segment_length (int): The desired length of each segment in tokens.
        
    Returns:
        list: A list of segments, each containing approximately `segment_length` tokens.
    """   
    # Split the story into tokens
    tokens = story.split()
    
    # Calculate the number of segments based on the desired segment length
    seg_count = math.ceil(len(tokens) / seg_length)
    
    # Length of each segment in tokens more equally distributed
    distrib_length = math.ceil(len(tokens) / seg_count) 
    
    # Create segments of approximately equal length
    # This will ensure that the segments are more evenly distributed in terms of token count
    segments = []
    for i in range(0, len(tokens), distrib_length):
        segment_tokens = tokens[i:i + distrib_length]
        segment = ' '.join(segment_tokens)
        if segment.strip():  # Add only non-empty segments
            segments.append(segment.strip())

    return segments

In [16]:
# Read an annotated story and return the segments.
def ann_segment_story(story):
    """
    Reads a text-tiling segmented story from a string and returns the segments.
    
    Args:
        story (str): The full text of the story.
        
    Returns:
        list: A list of segments extracted from the story.
    """
    segments = []
    # Split the story into parts based on the ANN_SEG label and ANN_SEP separator
    parts = story.split(ANN_SEP)
    for part in parts:
        # Skip the first line if it contains the ANN_SEG label
        if part.strip().startswith(ANN_SEG):
            part = '\n'.join(part.split('\n')[2:])  # Remove the line containing the annotation label
        if part.strip():  # Add only non-empty segments
            segments.append(part.strip())
    
    return segments

Define functions for computing segment similarity in stories using sentence-transformers.

In [27]:
# Compute cosine similarity for two segments
def compute_sgs(segment1, segment2, save_inter_res, inter_res_meth_file):
    """
    Computes the cosine similarity between two segments.
    
    Args:
        segment1 (str): The first segment of text.
        segment2 (str): The second segment of text.
        save_inter_res (bool): Whether to save intermediate results to a file.
        inter_res_meth_file (str): File name for saving method check as intermediate results.  
        
    Returns:
        float: The cosine similarity between the two segments.
    """
    if segment1 == None or segment2 == None:
        raise ValueError("Cannot compute segment similarity with empty segments")
    
    # Compute embeddings for the segments
    embeddings1 = model.encode(segment1, convert_to_tensor=True)
    embeddings2 = model.encode(segment2, convert_to_tensor=True)

     # Compute cosine similarity between the two segments
    cosine_sim = util.pytorch_cos_sim(embeddings1, embeddings2).item()

    # If save_inter_res, save word contributions as intermediate results
    if save_inter_res:
        with open(inter_res_meth_file, "a", encoding="utf-8") as f:
            f.write(f"{ANN_SEP[0:4]} Embeddings for segment 1: {embeddings1}\n{ANN_SEP[0:4]} Embeddings for segment 2: {embeddings2}\n")

    return cosine_sim

In [20]:
# Compute segment similarity for segments in a story
def compute_sgs_for_story(story, sgs_type='adjacent', seg_method='equal-length', save_inter_res=False, inter_res_seg_file=None, inter_res_meth_file=None):
    """
    Computes the cosine similarity for all segments in a story.
    
    Args:
        story (str): The full text of the story.
        sgs_type (str): The type of SGS to compute ('adjacent' or 'contextual').
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
        save_inter_res (bool): Whether to save intermediate results to a file.
        inter_res_seg_file (str): File name for saving segment check as intermediate results.
        inter_res_meth_file (str): File name for saving method check as intermediate results.
        
    Returns:
        list: A list of cosine similarity values for each pair of segments.
    """
    # Segment the story into smaller parts
    segments = ann_segment_story(story) if (seg_method == SEG_TEXT_TILING or seg_method == SEG_LLM_EPISODE) else \
    el_segment_story(story, seg_length=100) 

    # Compute SGS for each pair of adjacent or contextual segments
    sgs_values = []
    for i in range(len(segments) - 1):
        if sgs_type == SGS_CONTEXTUAL:
            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEP[0:4]} {ANN_SEG} 0-{i}: {' '.join(segments[0:i+1])}\n{ANN_SEP[0:4]} {ANN_SEG} {i+1}: {segments[i + 1]}\n{ANN_SEP}\n")
                with open(inter_res_meth_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEG} 0-{i} - {ANN_SEG} {i+1}\n")
            # For contextual SGS, compute SGS between the next segment and all previous segments
            sgs = compute_sgs(' '.join(segments[0:i+1]), segments[i + 1], save_inter_res, inter_res_meth_file)
        else:
            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEP[0:4]} {ANN_SEG} {i}: {segments[i]}\n{ANN_SEP[0:4]} {ANN_SEG} {i+1}: {segments[i + 1]}\n{ANN_SEP}\n")
                with open(inter_res_meth_file, "a", encoding="utf-8") as f:
                    f.write(f"{ANN_SEG} {i} - {ANN_SEG} {i+1}\n")
            # For adjacent SGS, compute SGS between the current segment and the next one  
            sgs = compute_sgs(segments[i], segments[i + 1], save_inter_res, inter_res_meth_file)
        sgs_values.append(sgs)
    
    return sgs_values

Define the function for analysing the stories and saving the results.

In [21]:
# Process a set of stories to compute SGS values for story segments and save results as CSV and PNG files
def process_sgs_stories(sgs_type, seg_method, save_inter_res):
    """
    Processes all stories in the input directory and saves the SGS results to the output directory.
    
    Args:
        sgs_type (str): The type of SGS to compute ('adjacent' or 'contextual').
        seg_method (str): The method used for segmenting the stories ('equal-length', 'text-tiling', 'llm-episodes').
        save_inter_res (bool): Whether to save intermediate results to a file.
    """
    cnt_processed = 0 # Counter for processed files
    sgs_avg_values = [] # List to store SGS averages for each story
    inp_file_names = [] # List to store input file names
    story_names = [] # List of story names
    num_chars = [] # Counter list for the number of characters processed
    num_toks = [] # Counter list for the number of tokens processed
    num_segs = [] # Counter list for the number of segments processed
    fn_sgs_type=FN_ADJACENT # File name suffix for SGS type, default is 'adjacent'
    fn_seg_meth=FN_EQUAL_LENGTH # File name suffix for segmentation method, default is 'equal-length'
    inp = input # Default input path for whole stories

    # Select the input path and the file name suffix based on the segmentation method
    if seg_method == SEG_TEXT_TILING:
        inp = input_seg_tt
        fn_seg_meth = FN_TEXT_TILING
    elif seg_method == SEG_LLM_EPISODE:
        inp = input_seg_llm
        fn_seg_meth = FN_LLM_EPISODE

    # Select the file name suffix based on the SGS type
    if sgs_type == SGS_CONTEXTUAL:
        fn_sgs_type = FN_CONTEXTUAL

    timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')  # Create a timestamp for the output files

    # File names for intermediate results
    inter_res_seg_file = None # Segment check file
    inter_res_meth_file = None # Probability distribution file 
    if save_inter_res: # Create the intermediate results files if saving is enabled
        inter_res_seg_file = os.path.join(output_inter_res, f"{INTER_RES_SEG}_{fn_sgs_type}_{fn_seg_meth}_{SGS_LABEL.lower()}_{timestamp}.txt")
        with open(inter_res_seg_file, "w", encoding="utf-8") as f:
            f.write(f"{INTER_RES_SEG.replace('_', ' ').capitalize()}: Segment Similarity (SGS): SGS type: {sgs_type}; segmentation method: {seg_method}\n{ANN_SEP}\n")
        inter_res_meth_file = os.path.join(output_inter_res, f"{INTER_RES_METH}_{fn_sgs_type}_{fn_seg_meth}_{SGS_LABEL.lower()}_{timestamp}.txt")
        with open(inter_res_meth_file, "w", encoding="utf-8") as f:
            f.write(f"{INTER_RES_METH.replace('_', ' ').capitalize()}: Segment similarity (SGS): SGS type: {sgs_type}; segmentation method: {seg_method}\n{ANN_SEP}\n")

    # Read the story files sequentially from the input folder.
    for file_path in Path(inp).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as input_file:
            story = input_file.read()
    
            # Print the name of the file being processed.
            print(f'Processing file: {file_path.name}')
            inp_file_names.append(file_path.name)  # Store the input file name

            if save_inter_res: # Save the intermediate results to a file
                with open(inter_res_seg_file, "a", encoding="utf-8") as f:
                    f.write(f"Processing file: {file_path.name}\n{ANN_SEP}\n")
                with open(inter_res_meth_file, "a", encoding="utf-8") as f:
                    f.write(f"Processing file: {file_path.name}\n{ANN_SEP}\n")

            if (seg_method == SEG_TEXT_TILING or seg_method == SEG_LLM_EPISODE):
                # Get the title of the story
                start = story.find(ANN_SEG + " 0:\n") + len(ANN_SEG + " 0:\n")
                end = story.find(ANN_SEP)
                story_name = story[start:end].strip()
                # Get the body of the story, excluding the title
                body = story[end+len(ANN_SEP):]
            else:   
                # Get the title of the story
                story_name = story.split('\n')[0]
                # Get the body of the story, excluding the title
                body = story[story.find('\n')+1:]
                
            # Compute SGS values for the story
            sgs_values = compute_sgs_for_story(body, sgs_type, seg_method, save_inter_res, inter_res_seg_file, inter_res_meth_file)
            sgs_avg = np.mean(sgs_values)  # SGS average for the story
            sgs_avg_values.append(sgs_avg)  # Store the SGS average for the story
            story_names.append(story_name)
            num_chars.append(len(body))  # Store the number of characters in the story
            num_toks.append(len(body.split()))  # Store the number of tokens in the story
            num_segs.append(len(sgs_values)+1)  # Store the number of segments in the story

            # Story SGS table
            df = pd.DataFrame({
                ANN_SEG_PAIR : range(1, len(sgs_values) + 1),
                SGS_LABEL : sgs_values,
                SGS_DEV : [sgs_val - sgs_avg for sgs_val in sgs_values],  # SGS deviation from the average by segment pair
                SGS_AVG : sgs_avg, # SGS average for the story
                SGS_STD : np.std(sgs_values)  # Standard deviation of SGS for the story
            })
            
            # Save the story DataFrame to a CSV file
            csv_path = os.path.join(output, OUT_CSV, f"{file_path.stem}_{fn_sgs_type}_{fn_seg_meth}_{SGS_LABEL.lower()}_{timestamp}.{OUT_CSV}")
            df.to_csv(csv_path, index=False)

            # Save the line plot as PNG
            plt.figure(figsize=(8, 4))
            plt.plot(df[ANN_SEG_PAIR], df[SGS_LABEL], marker='o')
            plt.xlabel(ANN_SEG_PAIR)
            plt.ylabel('Cosine Similarity')
            plt.title(f'Cosine similarity across {sgs_type} {seg_method.replace('llm', 'LLM')} segments: {story_name}', wrap=True, y = 1.02)
            plt.grid(True)
            plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))  # Ensure integer ticks
            plt.xlim(left=0)  # Start x-axis from 0
            # Set x-ticks to include the first and last segment numbers
            ticks = np.unique(np.concatenate(([df[ANN_SEG_PAIR].min()], plt.gca().get_xticks(), [df[ANN_SEG_PAIR].max()])))
            plt.xticks(ticks.astype(int))  # Set ticks (as integers)
            plt.tight_layout(rect=[0, 0, 1, 0.98])
            png_path = os.path.join(output, OUT_PNG, f"{file_path.stem}_{fn_sgs_type}_{fn_seg_meth}_{SGS_LABEL.lower()}_{timestamp}.{OUT_PNG}")
            plt.savefig(png_path)
            plt.close()

            cnt_processed += 1

    sgs_avg_by_run = np.mean(sgs_avg_values)  # Average SGS accross the run

    # Summary DataFrame for the run
    df_avg = pd.DataFrame({
        'ID': range(1, len(sgs_avg_values) + 1),
        'File': inp_file_names,  # Input file names 
        'Story': story_names,
        'Characters': num_chars,  # Number of characters by story
        'Tokens': num_toks,  # Number of tokens by story  
        'Segments': num_segs,  # Number of segments by story
        SGS_AVG: sgs_avg_values,  # SGS average by story
        SGS_DEV: [sgs_val - sgs_avg_by_run for sgs_val in sgs_avg_values],  # SGS deviation from the average by story
        SGS_AVG + ' Run': sgs_avg_by_run, # SGS average by run
        SGS_STD + ' Run': np.std(sgs_avg_values)  # Standard deviation of SGS by run
    })

    # Save the summary DataFrame to a CSV file
    csv_avg_path = os.path.join(output, OUT_CSV, f"summary_{fn_sgs_type}_{fn_seg_meth}_{SGS_LABEL.lower()}_{timestamp}.{OUT_CSV}")
    df_avg.to_csv(csv_avg_path, index=False)
    
    # Print the number of files saved in the output folder and the summary file name.
    end_proc_str = f'End processing: {cnt_processed} files saved to the {Path(output).name} folder.'
    print(end_proc_str)
    if save_inter_res:
        with open(inter_res_seg_file, "a", encoding="utf-8") as f:
            f.write(f"{end_proc_str}\n")
        with open(inter_res_meth_file, "a", encoding="utf-8") as f:
            f.write(f"{end_proc_str}\n")

    print(f'Summary of SGS values across the run saved to the {Path(csv_avg_path).name} file.')

Validate input, read the stories, analyse them, and save the results.  

In [ ]:
# Choose the SGS type and segmentation method
sgs_type = SGS_ADJACENT # SGS_ADJACENT or SGS_CONTEXTUAL
seg_method = SEG_EQUAL_LENGTH  # SEG_EQUAL_LENGTH or SEG_TEXT_TILING or SEG_LLM_EPISODE
# Set the flag to save intermediate results
save_inter_res = False  # Set to True to save intermediate results for segment and method check, False otherwise

# Validate the SGS type
if sgs_type not in [SGS_ADJACENT, SGS_CONTEXTUAL]:
    raise ValueError("Invalid SGS type. Choose from ", SGS_ADJACENT, "or ", SGS_CONTEXTUAL, ".")

# Validate the segmentation method and set the input path accordingly
if seg_method not in [SEG_EQUAL_LENGTH, SEG_TEXT_TILING, SEG_LLM_EPISODE]:
    raise ValueError("Invalid segmentation method. Choose from ", SEG_EQUAL_LENGTH, ", ", SEG_TEXT_TILING, "or ", SEG_LLM_EPISODE, ".")

# Process the stories with the specified SGS type, segmentation method and flag for saving intermediate results
process_sgs_stories(sgs_type, seg_method, save_inter_res)